# A foundation model for condensed matter: water, ions, and superionic ice

*Adapted from an [atomistic-cookbook](https://atomistic-cookbook.org) recipe by
Paolo Pegolo ([@ppegolo](https://github.com/ppegolo)).*

A **foundation** (or *universal*) machine-learning interatomic potential is trained once, on
a broad and chemically diverse dataset, and is then meant to describe essentially any system
without being re-fitted. The companion `ethanol-finetune` notebook asks how far you can push
one on a single small molecule, and what you gain by specialising it. This notebook asks the
opposite question — the one you face in practice when there is **no reference data to
fine-tune on**, because the system is a liquid, or a solution, or sits at a pressure no
experiment can easily reach.

We take **PET-MAD-XS**, unchanged, and run molecular dynamics on three systems it was never
specifically fitted to:

| system | what it tests |
|---|---|
| liquid water at 400 K | the baseline: does it produce a *liquid* with the right structure? |
| NaCl in water | two new elements and long-range ionic solvation, with no new parameters |
| superionic water at 3000 K, ~130 GPa | an exotic phase where oxygen stays crystalline and hydrogen flows like a liquid |

and an optional fourth, an ethanol-water mixture, in an appendix.

The point to keep in mind throughout: **the potential never changes**. The only
system-specific line in any of the inputs below is the one mapping LAMMPS atom types onto
chemical elements. Everything else — the ions, the extreme pressure, the carbon — the model
handles because it was trained across the periodic table.

### How to work through this

Cells marked **Your turn** contain a `TODO`: a number to read off a plot, a prediction to
commit to before running the next cell, or a parameter to change. They are the point of the
session.

**What runs live and what does not.** The water, NaCl and ethanol boxes run live at 1 ps
each — about a minute and a half on a GPU. That is long enough to watch the dynamics and
check it is stable, and far too short for converged statistics. So for the *analysis* we use
longer trajectories (10 ps for water and NaCl, 20 ps for superionic) precomputed with
exactly the same inputs and kept outside the repository, in `/home/unito/trajectories/`. Section 2 makes the comparison between the
two explicit, because knowing how much trajectory you need is half of doing MD properly.

## Setup

We use PET-MAD-XS through [`metatomic`](https://docs.metatensor.org/metatomic/latest/index.html),
which exposes the same potential to several simulation engines (ASE, i-PI, LAMMPS,
GROMACS, ...) behind a common interface. Here we drive it from **LAMMPS**, with a single
chemistry-agnostic line in every input file:

    pair_style metatomic pet-mad-xs.pt device ${device}
    pair_coeff * * 1 8

The first line loads the model; the second is the element map — LAMMPS atom type 1 is
hydrogen ($Z=1$), type 2 is oxygen ($Z=8$). Adding sodium and chlorine later means adding
`11 17` to that line, and nothing else.

Two practical notes on the cell below:

* The cell below compiles the PET-MAD-XS checkpoint into `pet-mad-xs.pt`, the TorchScript
  file that `pair_style metatomic` actually loads, exactly as the ethanol notebook does. The
  checkpoint lives on the classroom machine at `/home/unito/pet-mad-models/pet-mad-xs-v1.6.0.ckpt`;
  running elsewhere, point `CKPT_PATH` at your own copy.
* `${device}` defaults to `cpu` inside each `.lmp` file; `run_lammps` overrides it to `cuda`
  whenever a GPU is visible, and only then adds the Kokkos GPU flags. Everything here works
  without a GPU, just more slowly.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

import ase.io
import chemiscope
import matplotlib.pyplot as plt
import numpy as np
import torch
from ase.geometry.rdf import get_rdf
from tqdm.auto import tqdm

CKPT_PATH = Path("/home/unito/pet-mad-models/pet-mad-xs-v1.6.0.ckpt")
MODEL_PATH = Path("pet-mad-xs.pt")

# the long precomputed trajectories are too big to live in the repository; on the classroom
# machine they are here. Running elsewhere, point this at your own copy.
TRAJ_DIR = Path("/home/unito/trajectories")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {DEVICE}")

if not MODEL_PATH.exists():
    subprocess.run(
        [sys.executable, "-m", "metatrain", "export", str(CKPT_PATH), "-o", str(MODEL_PATH)],
        check=True,
    )
print(f"model ready at {MODEL_PATH}")


def run_lammps(input_file, nsteps=None):
    # `nsteps` overrides the default in the .lmp file (which declares it index-style
    # precisely so that it can be overridden from here)
    command = ["lmp", "-var", "device", DEVICE]
    if nsteps is not None:
        command += ["-var", "nsteps", str(nsteps)]
    if DEVICE == "cuda":
        # Kokkos GPU acceleration; these flags are meaningless (and harmful) on a CPU-only
        # machine, which is why they are added only when a GPU was actually found
        command += ["-k", "on", "g", "1", "-pk", "kokkos", "newton", "on", "neigh", "half",
                    "-sf", "kk"]
    command += ["-in", input_file]

    t0 = time.time()
    result = subprocess.run(command, capture_output=True, text=True)
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f"LAMMPS failed on {input_file}")
    steps = nsteps if nsteps is not None else float("nan")
    print(f"{input_file} finished in {elapsed:.1f} s ({1000 * elapsed / steps:.0f} ms/step)")


def load_traj(path):
    return ase.io.read(path, ":", format="lammps-dump-text")


def thermo_summary(thermo_file, equilibration=0.25):
    # columns written by the .lmp files: step temp pe etotal press vol
    data = np.loadtxt(thermo_file, skiprows=1)
    start = int(equilibration * len(data))
    step, temp, pe, etotal, press, vol = data[start:].T
    print(f"  {Path(thermo_file).name}: averages over the last {100 * (1 - equilibration):.0f}% "
          f"of the run")
    print(f"    temperature      : {temp.mean():7.1f} K  (fluctuating +- {temp.std():.0f} K)")
    print(f"    potential energy : {pe.mean():7.2f} eV")
    print(f"    pressure         : {press.mean() / 1e4:7.2f} GPa "
          f"(+- {press.std() / 1e4:.2f}; pressure is very noisy in a box this small)")
    return data

## 1. Liquid water at 400 K

64 water molecules in a cubic periodic box of side 12.52 A — a density of 0.97 g/cm$^3$,
i.e. real liquid water. The cell below prints the complete LAMMPS input driving it; it is
worth reading in full, because everything that follows is a variation on it.

Four choices worth understanding:

* **Ensemble.** `fix nve` propagates Newton's equations; `fix temp/csvr` adds a stochastic
  velocity-rescaling thermostat (Bussi-Donadio-Parrinello) on top. This samples the exact
  canonical distribution while disturbing the dynamics far less than a Langevin thermostat,
  which matters because we want to measure *transport* (diffusion) later, not just structure.
* **Temperature.** 400 K rather than 300 K. The electronic-structure reference PET-MAD was
  trained on (r2SCAN) over-structures liquid water and pushes its melting point up by a few
  tens of kelvin — a well-known property of most DFT functionals for water. Running warm
  keeps the system reliably liquid rather than sluggish or glassy.
* **Timestep.** 0.5 fs, set by the O-H stretch — the same argument as in the ethanol
  notebook, where you can watch what happens if you ignore it.
* **Periodic boundary conditions.** 64 molecules is not a beaker of water; PBC replicate the
  box infinitely so that no molecule sits at a surface. The price is that no correlation
  longer than half the box (6.26 A here) is meaningful, which will matter in section 2.

In [ ]:
print(Path("in_water_nvt.lmp").read_text())

In [ ]:
LIVE_STEPS = 2000  # TODO: 2000 steps x 0.5 fs = 1 ps. Lower it if you are short on time

run_lammps("in_water_nvt.lmp", nsteps=LIVE_STEPS)
water_thermo = thermo_summary("water_thermo.out")

The first check on any MD run: the thermostat should pull the temperature towards its target
(with large fluctuations — this is a small system), and the potential energy should settle
onto a plateau rather than drifting. The initial transient, while the velocities randomised
at $t=0$ redistribute, is *equilibration* and must be discarded before averaging anything —
`thermo_summary` throws away the first quarter of the run for exactly this reason.

Do not be surprised if the average printed above sits somewhat *below* 400 K. The velocities
are initialised at the target, but half of that kinetic energy immediately flows into
potential energy as the molecules start to vibrate, and the thermostat then has to feed the
difference back in over several of its own relaxation times (`tdamp` = 25 fs here, pretty aggressive). In a
1 ps run that recovery is a visible fraction of the whole trajectory. In the 10 ps
precomputed run it averages out to 399 K — another way of saying that a short run is for
watching, not for measuring.

In [ ]:
def plot_thermo(thermo, dt_fs, label, temperature_target=None):
    time_ps = thermo[:, 0] * dt_fs / 1000
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), dpi=120)
    axes[0].plot(time_ps, thermo[:, 1], lw=0.8)
    if temperature_target is not None:
        axes[0].axhline(temperature_target, color="tab:red", lw=1.2,
                        label=f"target {temperature_target:.0f} K")
        axes[0].legend(fontsize=8)
    axes[0].set(xlabel="time (ps)", ylabel="temperature (K)", title=f"Temperature - {label}")
    axes[1].plot(time_ps, thermo[:, 2], lw=0.8, color="tab:green")
    axes[1].set(xlabel="time (ps)", ylabel="potential energy (eV)",
                title=f"Potential energy - {label}")
    fig.tight_layout()
    plt.show()


plot_thermo(water_thermo, dt_fs=0.5, label="liquid water", temperature_target=400)

In [ ]:
water_live = load_traj("water_traj.lammpstrj")
for frame in water_live:
    frame.wrap()  # fold atoms back into the box, for display only

print(f"live trajectory: {len(water_live)} frames, "
      f"{len(water_live[0])} atoms, box {water_live[0].get_cell()[0, 0]:.2f} A")

chemiscope.show(
    structures=water_live,
    mode="structure",
    settings=chemiscope.quick_settings(
        trajectory=True,
        structure_settings={"playbackDelay": 20, "unitCell": True},
    ),
)

## 2. Structure: the radial distribution function

Watching the movie tells you the liquid is a liquid. To say anything quantitative we need a
*structural observable*, and the standard one is the **radial distribution function** $g(r)$.
Between two species A and B it measures how the local density of B around an A atom compares
with the average density of B:

$$g_{AB}(r) = \frac{1}{4\pi r^2 \rho_B} \left\langle \sum_{i \in A} \sum_{j \in B, j \neq i}
\delta(r - r_{ij}) \right\rangle$$

In practice it is a histogram of interatomic distances, normalised by the volume of each
spherical shell $4\pi r^2 dr$ and by the bulk density, so that a completely uncorrelated
(ideal-gas) arrangement gives $g(r) = 1$ everywhere. Everything interesting is a deviation
from 1: a tall first peak is a well-defined neighbour shell, and the **first minimum** after
it is the natural boundary of that shell.

Integrating up to that minimum gives the **coordination number** — how many neighbours a
particle has:

$$n_{AB} = 4\pi \rho_B \int_0^{r_\mathrm{min}} g_{AB}(r)\, r^2 \, dr$$

which is equivalent to simply counting neighbours within $r_\mathrm{min}$ frame by frame,
which is what we do below because it also gives the fluctuation.

**One hard limit.** In a periodic box of edge $L$ you cannot measure any correlation beyond
$L/2$ — past that, the spherical shell no longer fits inside the box and you start counting
periodic images of atoms you have already counted. Here $L/2 = 6.26$ A, so we stop at 6 A.

In [ ]:
def average_rdf(traj, elements, rmax=6.0, nbins=120, stride=1, desc=None):
    # g(r) averaged over frames. rmax must stay below L/2 (see the note above).
    accumulated = []
    frames = traj[::stride]
    for frame in tqdm(frames, desc=desc or f"g(r) {elements[0]}-{elements[1]}", leave=False):
        g, r = get_rdf(frame, rmax, nbins, elements=elements)
        accumulated.append(g)
    return r, np.mean(accumulated, axis=0)


def first_minimum(r, g):
    # the first minimum of g(r) after its first peak: the edge of the first shell
    peak = int(np.argmax(g))
    tail = g[peak:int(0.85 * len(g))]
    return float(r[peak + int(np.argmin(tail))])


def coordination_number(traj, center, cutoff, other="O", stride=1, desc=None):
    # neighbours of type `other` within `cutoff` of each `center` atom, per frame
    symbols = np.array(traj[0].get_chemical_symbols())
    i_center = np.where(symbols == center)[0]
    i_other = np.where(symbols == other)[0]
    counts = []
    for frame in tqdm(traj[::stride], desc=desc or f"n({center}-{other})", leave=False):
        distances = frame.get_all_distances(mic=True)[i_center][:, i_other]
        counts.append((distances < cutoff).sum(axis=1).mean())
    counts = np.array(counts)
    if center == other:
        counts -= 1  # an atom is not its own neighbour
    return counts

### How much trajectory do you actually need?

Here is the lesson that no amount of theory conveys as well as one plot. Below we compute the
oxygen-oxygen $g(r)$ twice: once from the 1 ps run you just did live, and once from a 10 ps
run produced with the *identical* input file, from `TRAJ_DIR`.

In [ ]:
water_long = load_traj(TRAJ_DIR / "water_traj.lammpstrj")
print(f"live run: {len(water_live)} frames (1 ps)   |   precomputed: {len(water_long)} frames (10 ps)")

r_short, g_short = average_rdf(water_live, ["O", "O"], desc="g_OO live (1 ps)")
r_long, g_long = average_rdf(water_long, ["O", "O"], stride=2, desc="g_OO precomputed (10 ps)")

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=120)
ax.plot(r_short, g_short, lw=1, color="tab:grey", label="live run, 1 ps")
ax.plot(r_long, g_long, lw=1.8, color="tab:blue", label="precomputed, 10 ps")
ax.axhline(1.0, color="k", lw=0.7, ls=":")
ax.set(xlabel=r"$r$ ($\AA$)", ylabel=r"$g_\mathrm{OO}(r)$",
       title="Oxygen-oxygen radial distribution function of liquid water")
ax.legend()
fig.tight_layout()
plt.show()

print(f"first peak at  {r_long[np.argmax(g_long)]:.2f} A   (experiment: 2.8 A)")
print(f"first minimum at {first_minimum(r_long, g_long):.2f} A")

Both curves show the same physics — a sharp first peak near 2.8 A (the hydrogen-bonded
neighbour shell, in excellent agreement with X-ray and neutron diffraction), a first minimum
near 3.5 A, and a weaker second shell — but the 1 ps curve is visibly noisier. With 64
molecules and 200 frames there are simply not enough independent samples in the far shells,
where the shell volume grows as $r^2$ but the statistics do not.

This is the single most common way to fool yourself in MD: a short run gives an answer, and
the answer looks plausible, and it is wrong at the few-percent level in ways that no
internal check reveals. The only honest tests are to run longer and see whether the answer
moves, and to quote an error bar.

> **Your turn.** Read the first minimum off the 10 ps curve and use it as the first-shell
> cutoff below. Before running the count: liquid water is often described as *tetrahedral*,
> which would mean 4 neighbours. Do you expect more or fewer than 4 at 400 K, and why?

In [ ]:
OO_CUTOFF = 3.5  # TODO: read the first minimum off YOUR plot above and put it here

n_oo_long = coordination_number(water_long, "O", OO_CUTOFF, "O")
n_oo_short = coordination_number(water_live, "O", OO_CUTOFF, "O")

print(f"first-shell O neighbours within {OO_CUTOFF} A")
print(f"   precomputed (10 ps): {n_oo_long.mean():.2f}  (fluctuating +- {n_oo_long.std():.2f})")
print(f"   live run    ( 1 ps): {n_oo_short.mean():.2f}  (fluctuating +- {n_oo_short.std():.2f})")

You should find roughly **5** — more than the 4 of a perfect tetrahedral network. Ice is
tetrahedral; liquid water at 400 K is a *broken* tetrahedral network, where the fifth
neighbour is an interstitial molecule that has not yet found a hydrogen bond. The number
also depends on where you put the cutoff, which is a general feature of coordination
numbers: they are only as well defined as the minimum you integrate to.

Notice how much better behaved this quantity is than $g(r)$ itself: the 1 ps and 10 ps runs
differ by only a few percent, while their $g(r)$ curves differ visibly everywhere. An average
over a *volume* pools all the pairs inside the shell, whereas $g(r)$ resolves them into thin
bins that each get a fraction of the data. Choosing observables that converge quickly is a
real skill in simulation — and quoting the ones that do not without an error bar is a real
way to get burned.

## 3. Adding ions: NaCl in water

Now two new elements, with **no new parameters and no retraining**: we dissolve 2 Na+ and
2 Cl- in 60 water molecules (184 atoms, a fairly concentrated ~1.8 M solution). The only
change in the entire input is the element map, which grows from `pair_coeff * * 1 8` to
`pair_coeff * * 1 8 11 17`.

This is where a foundation model earns its keep. Fitting a classical force field to ion-water
interactions means new parameters, fitted to new reference data, checked against new
experiments — for every ion. Here it is one line.

> **Your turn — predict first.** Na$^+$ and Cl$^-$ carry opposite charges of the same
> magnitude, but Cl$^-$ is a much larger ion (ionic radius ~1.8 A versus ~1.0 A). Which one
> will hold its first hydration shell *closer*, and which will hold *more* water molecules in
> it? Write down both answers before running the next cells.

In [ ]:
run_lammps("in_nacl_nvt.lmp", nsteps=LIVE_STEPS)
nacl_thermo = thermo_summary("nacl_thermo.out")
plot_thermo(nacl_thermo, dt_fs=0.5, label="NaCl solution", temperature_target=400)

With only two ions of each kind in the box, a 1 ps run gives hopeless statistics for an
ion-water $g(r)$ — four ions sampled for 200 frames. So from here on we analyse the 10 ps
precomputed trajectory. (The live run above is there to show you the machinery works and what
it costs; the science needs the longer one.)

In [ ]:
nacl_long = load_traj(TRAJ_DIR / "nacl_traj.lammpstrj")
for frame in nacl_long:
    frame.wrap()

shells = {}
fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=120)
for ion, color in (("Na", "tab:orange"), ("Cl", "tab:green")):
    r_ion, g_ion = average_rdf(nacl_long, [ion, "O"])
    cutoff = first_minimum(r_ion, g_ion)
    shells[ion] = cutoff
    ax.plot(r_ion, g_ion, color=color, lw=1.6, label=f"{ion}-O")
    ax.axvline(cutoff, color=color, ls="--", lw=0.9)
    print(f"{ion}-O: first peak at {r_ion[np.argmax(g_ion)]:.2f} A, "
          f"first minimum at {cutoff:.2f} A")

ax.axhline(1.0, color="k", lw=0.7, ls=":")
ax.set(xlabel=r"$r$ ($\AA$)", ylabel=r"$g(r)$",
       title="Ion-oxygen radial distribution (dashed = first-shell boundary)")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
NACL_STRIDE = 1  # every 2nd frame: plenty for an average, and half the work
nacl_frames = nacl_long[::NACL_STRIDE]

n_na = coordination_number(nacl_frames, "Na", shells["Na"], "O")
n_cl = coordination_number(nacl_frames, "Cl", shells["Cl"], "O")
print(f"Na+ : {n_na.mean():.2f} +- {n_na.std():.2f} waters within {shells['Na']:.2f} A")
print(f"Cl- : {n_cl.mean():.2f} +- {n_cl.std():.2f} waters within {shells['Cl']:.2f} A")

# how sensitive is the answer to where we draw the boundary?
print("\nsensitivity to the cutoff (this is why coordination numbers need a stated cutoff):")
for ion, counts_cutoff in (("Na", shells["Na"]), ("Cl", shells["Cl"])):
    row = [f"{coordination_number(nacl_long, ion, c, 'O', stride=10).mean():.2f}"
           for c in (counts_cutoff - 0.3, counts_cutoff, counts_cutoff + 0.3)]
    print(f"   {ion}+/-  cutoff {counts_cutoff - 0.3:.2f}/{counts_cutoff:.2f}/"
          f"{counts_cutoff + 0.3:.2f} A  ->  n = {' / '.join(row)}")

The model, with no ion-specific input whatsoever, reproduces the textbook picture:
Na+ binds its water molecules **tightly and close** (first peak near 2.4 A), while
Cl- holds a **looser, wider, more populated** shell (first peak near 3.1 A). The small,
dense charge of Na+ pulls oxygens in hard; the diffuse Cl- makes room for more
neighbours at a greater distance. Both are in reasonable agreement with neutron-diffraction
estimates (Na+ around 5-6, Cl- around 6-8, depending on concentration — and on the
cutoff, as the table above makes uncomfortably clear).

Treat these particular numbers with caution: four ions in a 10 ps trajectory is thin
sampling, and the fluctuation printed next to each mean is not a proper error bar on the mean
(consecutive frames are strongly correlated).

In [ ]:
# 0.5 fs timestep, dumped every 10 steps, then subsampled by NACL_STRIDE
time_ps = np.arange(len(nacl_long))[::NACL_STRIDE] * 0.0005 * 10
VIEW_STRIDE = 2  # chemiscope embeds every frame it is given into the notebook file

chemiscope.show(
    structures=nacl_frames[::VIEW_STRIDE],
    properties={
        "time [ps]": {"target": "structure", "values": time_ps[::VIEW_STRIDE]},
        "Na+ first-shell waters": {"target": "structure", "values": n_na[::VIEW_STRIDE]},
        "Cl- first-shell waters": {"target": "structure", "values": n_cl[::VIEW_STRIDE]},
    },
    mode="default",
    settings=chemiscope.quick_settings(
        trajectory=True,
        x="time [ps]",
        y="Na+ first-shell waters",
        structure_settings={"playbackDelay": 20, "unitCell": True},
        map_settings={"markerOutline": False},
    ),
)

Switch the y-axis dropdown between the two ions and watch the coordination number jump
between integers as individual water molecules enter and leave. That discreteness is the
physical reality behind the smooth average we quoted: a hydration shell is not a static cage
but a population in constant exchange.

## 4. Extreme conditions: superionic water

<img src="pics/superionic.png" width=600 height=250 >

Deep inside Uranus and Neptune, water is believed to exist as a **superionic** phase: the
oxygen atoms stay locked on a crystal lattice while the protons melt and flow through it like
a liquid. The material is simultaneously a solid (oxygen) and a liquid (hydrogen), and
because the mobile species is charged, it is an ionic conductor — which is one of the leading
explanations for the strange, off-centre magnetic fields of both planets.

Reaching it takes roughly 3000 K and ~130 GPa. This is exactly the regime where a foundation
model has an advantage that is hard to overstate: classical water force fields are fitted
around ambient conditions and are simply **undefined** here, and most of them cannot break an
O-H bond at all, so a proton could never hop. A model trained on a broad DFT dataset is
reactive by construction — bonds break and form because the underlying reference does.

Our box is 128 water molecules (384 atoms) on an ice-X-like lattice at a density of
3.20 g/cm$^3$, held at 3000 K and fixed volume for 20 ps. Three things change relative to the
runs above: much higher temperature, a shorter timestep (0.2 fs — atoms move much faster at
3000 K), and a much longer run, because diffusion is slow to measure and we need hydrogen to
visibly cross many lattice sites.

This one is **far too expensive to run live** — 100,000 steps on a box twice the size of the
water one, several hours on a good GPU. The trajectory in `TRAJ_DIR` was produced with
exactly the input printed below; read it against the water input from section 1 and note that
the only real differences are the temperature, the timestep and the length.

In [ ]:
print(Path("in_superionic_nvt.lmp").read_text())

In [ ]:
superionic_thermo = np.loadtxt("trajs/superionic_thermo.out", skiprows=1)
half = len(superionic_thermo) // 2  # average over the second half only

n_water = 128
volume_A3 = superionic_thermo[0, 5]
density = n_water * 18.015 / 6.02214e23 / (volume_A3 * 1e-24)  # g/cm^3

print("conditions actually sampled in the precomputed run:")
print(f"   temperature {superionic_thermo[half:, 1].mean():7.0f} K")
print(f"   pressure    {superionic_thermo[half:, 4].mean() / 1e4:7.1f} GPa")
print(f"   density     {density:7.2f} g/cm^3   "
      f"(liquid water at ambient: 1.00)")

### A quick definition: the mean-squared displacement

Structure told us about the liquid; to separate a solid from a liquid we need **dynamics**.
The mean-squared displacement of a species tracks how far its atoms wander from where they
started:

$$\mathrm{MSD}(t) = \left\langle \frac{1}{N} \sum_{i=1}^{N}
\left| \mathbf{r}_i(t) - \mathbf{r}_i(0) \right|^2 \right\rangle$$

Its long-time behaviour distinguishes the two states of matter cleanly:

* in a **solid**, every atom rattles around a fixed lattice site, so the MSD saturates at a
  constant — the mean-square rattling amplitude;
* in a **liquid**, atoms wander without bound and the MSD grows *linearly*,
  $\mathrm{MSD}(t) = 6Dt$ in three dimensions, where $D$ is the self-diffusion coefficient.
  This is Fick's law seen from inside a single particle's trajectory.

So the signature of the superionic phase is simply: **hydrogen liquid-like, oxygen
solid-like, in the same simulation at the same time.**

One implementation note that matters more than it looks. Written as above, the MSD uses only
$t=0$ as its reference, which throws away almost all the data: a 20 ps trajectory contains
many independent 5 ps windows, not one. Averaging over **all time origins** uses them all,
and costs nothing but a loop. We do that below, and only use lag times up to half the
trajectory, beyond which too few windows remain for the average to mean anything.

The trajectory is dumped with *unwrapped* coordinates (`xu yu zu` in the LAMMPS input), so
displacements can be taken directly — with wrapped coordinates an atom crossing the box edge
would register a spurious jump of one box length.

In [ ]:
def mean_squared_displacement(traj, species, max_lag_fraction=0.5):
    symbols = np.array(traj[0].get_chemical_symbols())
    mask = symbols == species
    positions = np.array([frame.get_positions()[mask] for frame in traj])  # (frames, N, 3)

    n_frames = len(positions)
    n_lags = int(max_lag_fraction * n_frames)
    msd = np.zeros(n_lags)
    for lag in tqdm(range(n_lags), desc=f"MSD {species}", leave=False):
        displacement = positions[lag:] - positions[:n_frames - lag]
        msd[lag] = np.mean(np.sum(displacement ** 2, axis=-1))
    return msd


def diffusion_coefficient(time_ps, msd, fit_from=0.4, fit_to=0.9):
    # D from the slope of the linear part: MSD = 6 D t. 1 A^2/ps = 1e-4 cm^2/s
    lo, hi = int(fit_from * len(time_ps)), int(fit_to * len(time_ps))
    slope = np.polyfit(time_ps[lo:hi], msd[lo:hi], 1)[0]
    return slope / 6 * 1e-4


superionic_traj = load_traj(TRAJ_DIR / "superionic_traj.lammpstrj")
DT_PS, DUMP_EVERY = 0.0002, 50  # 0.2 fs timestep, dumped every 50 steps = 10 fs per frame

msd = {species: mean_squared_displacement(superionic_traj, species) for species in ("H", "O")}
lag_ps = np.arange(len(msd["H"])) * DT_PS * DUMP_EVERY
print(f"trajectory: {len(superionic_traj)} frames = "
      f"{len(superionic_traj) * DT_PS * DUMP_EVERY:.0f} ps; MSD up to {lag_ps[-1]:.0f} ps")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), dpi=120)

for species, color in (("H", "tab:blue"), ("O", "tab:red")):
    axes[0].plot(lag_ps, msd[species], color=color, label=species)
axes[0].set(xlabel="lag time (ps)", ylabel=r"MSD ($\AA^2$)",
            title="Superionic water: 3000 K, ~130 GPa")
axes[0].legend()

for species, color in (("H", "tab:blue"), ("O", "tab:red")):
    axes[1].loglog(lag_ps[1:], msd[species][1:], color=color, label=species)

# a pure slope of 1 (Fickian diffusion) to compare against -- a guide, not a fit
guide_t = lag_ps[len(lag_ps) // 4:]
guide_ref = msd["H"][len(lag_ps) // 4]
axes[1].loglog(guide_t, guide_ref * guide_t / guide_t[0], "k:", lw=1)
axes[1].text(guide_t[len(guide_t) // 2], guide_ref * 2.4, r"$\propto t$", fontsize=9)
axes[1].axhline(msd["O"][len(msd["O"]) // 2:].mean(), color="tab:red", ls="--", lw=1)
axes[1].text(lag_ps[2], msd["O"][len(msd["O"]) // 2:].mean() * 1.3,
             "oxygen: trapped on its lattice site", fontsize=8, color="tab:red")
axes[1].set(xlabel="lag time (ps)", ylabel=r"MSD ($\AA^2$)",
            title="Same data, log-log: the two regimes separate")
axes[1].legend()

fig.tight_layout()
plt.show()

print("self-diffusion coefficients from the linear part:")
for species in ("H", "O"):
    print(f"   D({species}) = {diffusion_coefficient(lag_ps, msd[species]):.2e} cm^2/s")
print(f"\n   ratio D(H)/D(O) = "
      f"{diffusion_coefficient(lag_ps, msd['H']) / diffusion_coefficient(lag_ps, msd['O']):.0f}")

There it is, in one plot. On the linear axes hydrogen is a straight line through the origin —
Fickian diffusion, indistinguishable from a liquid — while oxygen is flat on the axis. The
log-log panel separates the regimes properly:

* **Oxygen** rises briefly and then flattens onto a plateau by about 0.1 ps, staying at
  ~0.25 A$^2$ for the entire 10 ps. That plateau *is* the mean-square rattling amplitude
  about a lattice site: excursions of half an angstrom, going nowhere. This is what "frozen
  on a lattice" looks like as a number.
* **Hydrogen** is already diffusing at the shortest lag time we can resolve. There is no
  visible caging plateau for it — at 3000 K the proton escapes its cage faster than our 10 fs
  dump interval, so by the first plotted point it is long gone. (Note that neither species
  shows the textbook ballistic $\propto t^2$ regime either: at this density collisions happen
  within a few femtoseconds, far below our sampling.)

The two species differ in mobility by roughly **four orders of magnitude** while sharing the
same box, the same temperature and the same potential. No parameter in the model was told
that hydrogen should melt and oxygen should not.

> **Your turn.** Compare the hydrogen diffusion coefficient printed above with liquid water's,
> which is about $2 \times 10^{-5}$ cm$^2$/s at room temperature. Is the proton in superionic
> ice moving *slower* than water molecules in a liquid, comparably, or faster? Then ask
> yourself the harder question: the oxygen $D$ is not exactly zero — is that real diffusion,
> or is it the tail of the rattling motion contaminating a straight-line fit? How would you
> tell the two apart?

### Watching the transition happen

The run above starts already superionic. To see the phase *form*, here is a second
precomputed trajectory in which the thermostat ramps from 300 K to 3000 K over 20 ps
(the same input with `fix temp/csvr 300 3000 ...`). Play it while watching the temperature on
the map: the oxygen lattice stays visibly ordered throughout, while the hydrogens progressively
break loose and begin to flow between sites.

In [ ]:
ramp_traj = load_traj(TRAJ_DIR / "superionic_ramp_traj.lammpstrj")
ramp_thermo = np.loadtxt("trajs/superionic_ramp_thermo.out", skiprows=1)

# chemiscope embeds every frame it is handed into the notebook file, and this trajectory has
# ~2000 of them; every 5th frame (50 fs apart) is still perfectly smooth to watch
VIEW_STRIDE = 1

chemiscope.show(
    structures=ramp_traj[::VIEW_STRIDE],
    properties={
        "time [ps]": {"target": "structure",
                      "values": ramp_thermo[::VIEW_STRIDE, 0] * DT_PS},
        "temperature [K]": {"target": "structure", "values": ramp_thermo[::VIEW_STRIDE, 1]},
    },
    mode="default",
    settings=chemiscope.quick_settings(
        trajectory=True,
        x="time [ps]",
        y="temperature [K]",
        structure_settings={"playbackDelay": 5, "unitCell": True},
        map_settings={"markerOutline": False},
    ),
)

## 5. Appendix: an organic mixture, ethanol-water

If time allows, one more system: 8 ethanol molecules mixed with 40 waters. Ethanol brings a
third element, carbon, which for LAMMPS means one more atom type and one more entry in the
element map (`pair_coeff * * 1 8 6`). The model handles the new C-H, C-C, C-O and
carbon-water interactions with nothing added — the kind of thing that is genuinely tedious to
parameterise in a classical force field.

The claim to test is that the two species **mix and hydrogen-bond to each other**. We count
hydrogen bonds with the standard geometric definition: an O-H$\cdots$O motif with the two
oxygens within 3.5 A and a donor-H-acceptor angle above 150 degrees, classifying each bond as
water-water or water-ethanol.

In [ ]:
run_lammps("in_ethanol_nvt.lmp", nsteps=LIVE_STEPS)
ethanol_thermo = thermo_summary("ethanol_thermo.out")

In [ ]:
def count_hbonds(traj, r_oo=3.5, angle_deg=150.0):
    # per-frame counts of (water-water, water-ethanol) hydrogen bonds
    symbols = np.array(traj[0].get_chemical_symbols())
    box = traj[0].get_cell()[0, 0]  # cubic box: minimum-image convention by hand
    i_o = np.where(symbols == "O")[0]
    i_h = np.where(symbols == "H")[0]
    i_c = np.where(symbols == "C")[0]
    heavy = np.concatenate([i_o, i_c])

    def minimum_image(a, b):
        d = a[:, None, :] - b[None, :, :]
        return d - np.round(d / box) * box

    # an oxygen bonded to a carbon belongs to an ethanol; the others are water
    first_frame = traj[0].get_positions()
    is_ethanol_o = (np.linalg.norm(minimum_image(first_frame[i_o], first_frame[i_c]),
                                   axis=-1) < 1.7).any(axis=1)
    cos_threshold = np.cos(np.radians(angle_deg))

    water_water, water_ethanol = [], []
    for frame in tqdm(traj, desc="counting H-bonds", leave=False):
        p = frame.get_positions()
        # assign each H to its nearest heavy atom; keep those covalently bound to an O
        distance_to_heavy = np.linalg.norm(minimum_image(p[i_h], p[heavy]), axis=-1)
        nearest = np.argmin(distance_to_heavy, axis=1)
        is_hydroxyl = ((nearest < len(i_o))
                       & (distance_to_heavy[np.arange(len(i_h)), nearest] < 1.3))
        h_donor, o_donor = i_h[is_hydroxyl], i_o[nearest[is_hydroxyl]]

        v_hd = p[o_donor] - p[h_donor]
        v_hd -= np.round(v_hd / box) * box          # H -> its own oxygen
        v_ha = -minimum_image(p[h_donor], p[i_o])   # H -> every candidate acceptor
        r_da = np.linalg.norm(minimum_image(p[o_donor], p[i_o]), axis=-1)
        cos_angle = ((v_hd[:, None, :] * v_ha).sum(-1)
                     / (np.linalg.norm(v_hd, axis=-1)[:, None]
                        * np.linalg.norm(v_ha, axis=-1) + 1e-9))
        is_hbond = (r_da < r_oo) & (cos_angle < cos_threshold)

        donor_is_ethanol = is_ethanol_o[nearest[is_hydroxyl]][:, None]
        acceptor_is_ethanol = is_ethanol_o[None, :]
        water_water.append(int((is_hbond & ~donor_is_ethanol & ~acceptor_is_ethanol).sum()))
        water_ethanol.append(int((is_hbond & (donor_is_ethanol ^ acceptor_is_ethanol)).sum()))
    return np.array(water_water), np.array(water_ethanol)


ethanol_traj = load_traj("ethanol_traj.lammpstrj")
hb_water_water, hb_water_ethanol = count_hbonds(ethanol_traj)
print(f"water-water   H-bonds: {hb_water_water.mean():.1f} +- {hb_water_water.std():.1f}")
print(f"water-ethanol H-bonds: {hb_water_ethanol.mean():.1f} +- {hb_water_ethanol.std():.1f}")

> **Your turn.** The 3.5 A / 150 degree criterion is a *convention*, not a law of nature —
> hydrogen bonding is a continuum, and any sharp definition draws an arbitrary line through
> it. Re-run `count_hbonds` with `r_oo=3.2` or `angle_deg=130` and see how much the counts
> move. Then decide: is the water-ethanol *fraction* of all hydrogen bonds more or less
> robust to this choice than the absolute count? Which of the two would you put in a paper?

In [ ]:
ethanol_view = ethanol_traj[::2]
for frame in ethanol_view:
    frame.wrap()
time_ps = np.arange(len(ethanol_traj))[::2] * 0.0005 * 10

chemiscope.show(
    structures=ethanol_view,
    properties={
        "time [ps]": {"target": "structure", "values": time_ps},
        "water-ethanol H-bonds": {"target": "structure", "values": hb_water_ethanol[::2]},
        "water-water H-bonds": {"target": "structure", "values": hb_water_water[::2]},
    },
    mode="default",
    settings=chemiscope.quick_settings(
        trajectory=True,
        x="time [ps]",
        y="water-ethanol H-bonds",
        structure_settings={"playbackDelay": 20, "unitCell": True},
        map_settings={"markerOutline": False},
    ),
)

## Where to go next

Everything above used one foundation model, out of the box, changing only the system and
never the potential — across a liquid, an electrolyte, a planetary-interior phase and an
organic mixture. That is often enough for a qualitative picture, a starting structure, or a
screening study.

When it is *not* enough, the answer is fine-tuning: a small, targeted set of reference
calculations that specialises the universal model to your system, at a fraction of the cost
of training from scratch. The companion **`ethanol-finetune`** notebook does exactly that,
and also shows how to find out whether your model is good enough for the question you are
asking — which, as section 3 hinted, is the harder half of the problem.

* [PET-MAD tutorial](https://atomistic-cookbook.org/examples/pet-mad/pet-mad.html) — running
  PET-MAD yourself with ASE, i-PI and LAMMPS.
* [Fine-tuning PET-MAD](https://atomistic-cookbook.org/examples/pet-finetuning/pet-ft.html)
  — specialising a foundation model to a target system.
* [Mendeleev's nano-soup](https://atomistic-cookbook.org/examples/mendeleev/mendeleev.html)
  — the same model pushed to its limit: a 102-element nanoparticle with replica-exchange MD
  and Monte Carlo atom swaps.